In [1]:
from __future__ import annotations

import json
import re
import time
from typing import TypedDict, Optional

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langgraph.graph import StateGraph, START, END

# ✅ 문제가 많던 langchain.retrievers 대신, AI 모델 라이브러리를 직접 사용합니다.
from sentence_transformers import CrossEncoder

from langchain_chroma import Chroma


In [3]:
# ---------------------------------------------------------------------------
# 설정값 (govfund_BuildDB.py 와 동일하게 맞춰야 합니다)
# ---------------------------------------------------------------------------

DB_DIR = "../chroma_govfund_db"
COLLECTION_NAME = "govfund_guide"
DB_DIR = "chroma_travel_db"
COLLECTION_NAME = "travel_guide"
EMBEDDING_MODEL = "bge-m3"        # 사용자 환경에 맞게 변경 (ollama pull bge-m3 필요)
OLLAMA_MODEL = "gemma4:e4b"      # 사용자 환경에 맞게 변경
RERANK_MODEL_NAME = "Dongjin-kr/ko-reranker"

TOP_K_PER_QUERY = 3          # Query 1개당 검색할 문서 수
POST_PROCESS_LIMIT = 8       # Post-Processing 후 Reranking으로 넘길 후보 문서 수
RERANK_TOP_N = 5             # Reranking 이후 최종 유지 문서 수
SCORE_THRESHOLD = 0.5        # relevance_score 최소 기준 (이 값 미만 제거)

CATEGORY_MAP = {
    "(공고문)_2026년도_중앙부처_및_지자체_창업지원사업_통합공고문(제2025-648호,_2025.12.19.).pdf": "notice",
}

In [4]:
try:
    embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL, base_url="http://10.8.0.1:11434")
except Exception as e:
    print(f"⚠️ Embeddings 초기화 실패: {e}")
try:
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0,base_url="http://10.8.0.1:11434")
except Exception as e:
    print(f"⚠️ LLM 초기화 실패: {e}")

try:
    # device="cpu": GPU를 Ollama(임베딩/LLM)와 동시에 점유하면
    # CUDA 컨텍스트 충돌로 ollama의 llama-server가 크래시할 수 있어 CPU로 고정
    reranker_model = CrossEncoder(RERANK_MODEL_NAME, device="cpu")
except Exception as e:
    reranker_model = None
    print(f"⚠️ Reranker 초기화 실패: {e}")

D:\Works\ai-playground\.venv\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
# 기존 vectorstore 로드

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=DB_DIR,
)